In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset , DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report , confusion_matrix

import warnings
warnings.filterwarnings("ignore")

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

if torch.cuda.is_available():
    print(f"Running on : {torch.cuda.get_device_name()}")
    print(f"Total Device : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")

print(f"Library Versions:")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")

print("\nEnvironment setup complete!")


cuda
Running on : NVIDIA GeForce RTX 4060 Laptop GPU
Total Device : 8.0 GB
Library Versions:
PyTorch: 2.11.0+cu128
NumPy: 2.4.1
Matplotlib: 3.10.8

Environment setup complete!


In [29]:
class SimpleCNN(nn.Module):
    """ Initlize the Layers of our CNNs """
    
    def __init__(self):
        super().__init__()
        # First Convolutional Block
        self.conv1 = nn.Conv2d(
            in_channels = 1,
            out_channels = 32,
            kernel_size = 3,
            stride = 1,
            padding = 1,
        )
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2 , stride=2) # 2x2 max pooling

        # Second Convolutional Block
        self.conv2 = nn.Conv2d(
            in_channels = 32,
            out_channels = 64,
            kernel_size = 3,
            stride = 1,
            padding = 1,
        )
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2 , stride=2)


        # Third Convolutional Block
        self.conv3 = nn.Conv2d(
            in_channels = 64,
            out_channels = 128,
            kernel_size = 3,
            stride = 1,
            padding = 1,
        )
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2 , stride=2)


        # Fully Connected Layer
        self.fc1 = nn.Linear(3*3*128 , 128)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128 , 10)

    

    def forward(self , x):
        """ Forward Pass through the Network """
        # Input Shape = [batch_size , 1 , 28 , 28]
        
        # 1st Convolutional Block
        x = self.conv1(x) # -> [batch_size , 32 , 28 , 28]
        x = self.relu1(x) # -> [batch_size , 32 , 28 , 28]
        x = self.pool1(x) # -> [batch_size , 32 , 14 , 14]

        # 2nd Convolutional Block
        x = self.conv2(x) # -> [batch_size , 64 , 14 , 14]
        x = self.relu2(x) # -> [batch_size , 64 , 14 , 14]
        x = self.pool2(x) # -> [batch_size , 64 , 7 , 7]

        # 3rd Convolutional Block
        x = self.conv3(x) # -> [batch_size , 128 , 7 , 7]
        x = self.relu3(x) # -> [batch_size , 128 , 7 , 7]
        x = self.pool3(x) # -> [batch_size , 128 , 3 , 3]

        x = x.view(x.size(0) , -1) # -> [batch_size , 3*3*128 = 1152]

        # Fully Connected Layers
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

    def feature_map(self , x):
        """ Extract intermediate feature map for visualization. """

        feature_maps = {}

        # First Block 
        x = self.conv1(x)
        feature_maps["conv1"] = x.detach().clone()
        x = self.relu1(x)
        x = self.pool1(x)
        feature_maps["pool1"] = x.detach().clone()

        # Second Block
        x = self.conv2(x)
        feature_maps["conv2"] = x.detach().clone()
        x = self.relu2(x)
        x = self.pool2(x)
        feature_maps["pool2"] = x.detach().clone()

        # Third Block
        x = self.conv3(x)
        feature_maps["conv3"] = x.detach().clone()
        x = self.relu3(x)
        x = self.pool3(x)
        feature_maps["pool3"] = x.detach().clone()

        return feature_maps


In [30]:
model = SimpleCNN()
print(model)

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1152, out_features=128, bias=True)
  (relu4): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters : {total_params}")

sample_input = torch.randn(1 , 1 , 28 , 28)
sample_output = model(sample_input)

Total Parameters : 241546


In [32]:
print(f"Input Shape : {sample_input.shape}")
print(f"Output Shape : {sample_output.shape}")

Input Shape : torch.Size([1, 1, 28, 28])
Output Shape : torch.Size([1, 10])


In [42]:
logits = sample_output.reshape(-1).detach().numpy()
classes = (sample_output.reshape(-1) > 0.5).float()
print(f"Output Values Logits : {logits}")
print(f"Classes : {classes}")

Output Values Logits : [ 0.10878858 -0.0266936  -0.01730811 -0.14230078 -0.0038314   0.07699438
 -0.03547343  0.07929878  0.07361799 -0.0076621 ]
Classes : tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
